In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from imports import *
from config import main_config, dir_config
from src.utils import pmf_utils

In [ ]:
compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

prior_colors = main_config.colors.prior_colors
block_colors = main_config.colors.block_colors

In [ ]:
with open(Path(processed_dir, 'sessions_metadata.csv'), 'r') as f:
    session_metadata = pd.read_csv(f)

## Chronometric functions per session (correct trials, equal vs unequal block)

In [ ]:
for _, session_row in session_metadata.iterrows():
    session_id = session_row["session_id"]
    prior_direction = (
        "L"
        if (session_row["prior_direction"] == "toRF" and session_row["RF_side"] == "L")
        or (session_row["prior_direction"] == "awayRF" and session_row["RF_side"] == "R")
        else "R"
    )

    trial_data = pd.read_csv(
        Path(compiled_dir, session_id, f"{session_id}_trial_cleaned.csv"), index_col=None
    )
    gp_trials = trial_data[trial_data.task_type == 1].reset_index(drop=True)
    valid = gp_trials[gp_trials.outcome >= 0].copy()

    coherence = valid.coherence.values
    target = valid.target.values.astype(int)
    choices = valid.choice.values.astype(int)
    outcomes = valid.outcome.values.astype(int)

    if session_row["RF_side"] == "L":
        target = 1 - target
        choices = 1 - choices

    signed_coherence = coherence * (target * 2 - 1)

    session_df = pd.DataFrame({
        "signed_coherence": signed_coherence,
        "choice": choices,
        "outcome": outcomes,
        "response_time": valid.reaction_time.values,
        "prob_toRF": valid.prob_toRF.values,
    })

    equal_data = session_df[session_df.prob_toRF == 50]
    unequal_data = session_df[session_df.prob_toRF != 50]

    # skip if either block is missing or has no correct trials
    if equal_data[equal_data.outcome == 1].empty or unequal_data[unequal_data.outcome == 1].empty:
        print(f"Skipping {session_id}: insufficient correct trials in one block")
        continue

    eq_coh, _, eq_rt_mean, _, eq_rt_sem = pmf_utils.get_chronometric_data(equal_data, outcome=1)
    uneq_coh, _, uneq_rt_mean, _, uneq_rt_sem = pmf_utils.get_chronometric_data(unequal_data, outcome=1)

    fig, ax = plt.subplots(figsize=(8, 5))

    color_eq = prior_colors["equal"]
    color_uneq = prior_colors[prior_direction]

    ax.plot(eq_coh, eq_rt_mean, marker="o", color=color_eq, linewidth=2, markersize=7, label="Equal block")
    ax.fill_between(eq_coh, eq_rt_mean - eq_rt_sem, eq_rt_mean + eq_rt_sem, color=color_eq, alpha=0.25)

    ax.plot(uneq_coh, uneq_rt_mean, marker="o", color=color_uneq, linewidth=2, markersize=7,
            label=f"Unequal block (prior {prior_direction})")
    ax.fill_between(uneq_coh, uneq_rt_mean - uneq_rt_sem, uneq_rt_mean + uneq_rt_sem, color=color_uneq, alpha=0.25)

    ax.set_xticks([-50, -20, -6, 0, 6, 20, 50])
    ax.set_xlabel("Coherence (%)", fontsize=14)
    ax.set_ylabel("Mean Reaction Time (ms)", fontsize=14)
    ax.set_title(f"{session_id}  |  prior: {prior_direction}", fontsize=14)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=12, frameon=False)
    ax.tick_params(labelsize=11)

    plt.tight_layout()
    plt.show()